# 2. Baseline Evaluation: Zero-Shot vs Chain-of-Thought (CoT)
**HLT 2026 - Reasoning with Verifiable Rewards**

This notebook evaluates the base model (**Qwen2.5-0.5B-Instruct** or 1.5B) on the fixed 200-puzzle benchmark test set before reinforcement learning:
1. **Method 1: Zero-Shot Direct** (direct answer with no explanation).
2. **Method 2: Chain-of-Thought (CoT)** (explicit step-by-step reasoning prompt, *Wei et al., 2022*).

> ⚡ **Hardware Requirement**:
> * **Recommended on Google Colab with free T4 GPU** (benchmark runs in ~2–3 minutes).
> * CPU execution is possible for 0.5B, but significantly slower.

## 1. Install Dependencies (Colab)

In [ ]:
!pip install -q transformers accelerate datasets pandas

## 2. Environment Setup & AST Verifier

In [ ]:
import os
import ast
import re
import json
import torch
import urllib.request
import pandas as pd
from typing import List, Dict, Tuple, Optional, Any
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Active compute device: {device}")
if device == "cuda":
    print("Detected GPU:", torch.cuda.get_device_name(0))

In [ ]:
class CountdownVerifier:
    """
    Deterministic and secure AST-based verifier for the Countdown arithmetic game.
    """
    ALLOWED_OPERATORS = {
        ast.Add: lambda a, b: a + b,
        ast.Sub: lambda a, b: a - b,
        ast.Mult: lambda a, b: a * b,
        ast.Div: lambda a, b: a / b if b != 0 else None,
        ast.FloorDiv: lambda a, b: a // b if b != 0 else None,
        ast.USub: lambda a: -a,
        ast.UAdd: lambda a: +a
    }

    ALLOWED_NODE_TYPES = (
        ast.Expression,
        ast.BinOp,
        ast.UnaryOp,
        ast.Constant,
        ast.Add,
        ast.Sub,
        ast.Mult,
        ast.Div,
        ast.FloorDiv,
        ast.USub,
        ast.UAdd
    )

    @staticmethod
    def extract_tags(text: str) -> Tuple[Optional[str], Optional[str]]:
        """Extract reasoning and final equation from XML-like tags <think> and <answer>."""
        think_match = re.search(r"<think>(.*?)</think>", text, re.DOTALL | re.IGNORECASE)
        thought = think_match.group(1).strip() if think_match else None

        answer_match = re.search(r"<answer>(.*?)</answer>", text, re.DOTALL | re.IGNORECASE)
        answer = answer_match.group(1).strip() if answer_match else None

        if not answer:
            answer_partial = re.search(r"<answer>(.*)", text, re.DOTALL | re.IGNORECASE)
            if answer_partial:
                answer = answer_partial.group(1).strip().split("\n")[0].strip()

        return thought, answer

    @classmethod
    def is_safe_ast(cls, tree: ast.AST) -> bool:
        """Verify that the AST contains ONLY allowed arithmetic nodes and operators."""
        for node in ast.walk(tree):
            if not isinstance(node, cls.ALLOWED_NODE_TYPES):
                return False
            if isinstance(node, ast.BinOp) and type(node.op) not in cls.ALLOWED_OPERATORS:
                return False
            if isinstance(node, ast.UnaryOp) and type(node.op) not in cls.ALLOWED_OPERATORS:
                return False
            if isinstance(node, ast.Constant):
                if isinstance(node.value, bool) or not isinstance(node.value, (int, float)):
                    return False
        return True

    @classmethod
    def evaluate_ast(cls, node: ast.AST) -> Optional[float]:
        """Recursively evaluate an AST node, allowing only basic arithmetic operations."""
        if isinstance(node, ast.Expression):
            return cls.evaluate_ast(node.body)

        elif isinstance(node, ast.Constant) and isinstance(node.value, (int, float)) and not isinstance(node.value, bool):
            return float(node.value)

        elif isinstance(node, ast.UnaryOp):
            op_func = cls.ALLOWED_OPERATORS.get(type(node.op))
            if not op_func:
                return None
            val = cls.evaluate_ast(node.operand)
            return op_func(val) if val is not None else None

        elif isinstance(node, ast.BinOp):
            op_func = cls.ALLOWED_OPERATORS.get(type(node.op))
            if not op_func:
                return None
            left_val = cls.evaluate_ast(node.left)
            right_val = cls.evaluate_ast(node.right)
            if left_val is None or right_val is None:
                return None
            try:
                res = op_func(left_val, right_val)
                return res
            except (ZeroDivisionError, OverflowError):
                return None

        return None

    @classmethod
    def extract_numbers_from_ast(cls, node: ast.AST) -> List[int]:
        """Extract all numbers used in the parsed AST expression."""
        numbers = []
        for n in ast.walk(node):
            if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)) and not isinstance(n.value, bool):
                if float(n.value).is_integer():
                    numbers.append(int(n.value))
                else:
                    numbers.append(n.value)
        return numbers

    @classmethod
    def verify(cls, available_numbers: List[int], target: int, completion: str) -> Dict[str, Any]:
        """
        Full verification of the model completion.
        Returns a dictionary containing formatting, validity, and correctness status.
        """
        thought, answer_str = cls.extract_tags(completion)

        result = {
            "has_format": bool(thought is not None and answer_str is not None),
            "thought": thought,
            "answer_str": answer_str,
            "valid_syntax": False,
            "valid_numbers": False,
            "used_numbers": [],
            "computed_value": None,
            "is_correct": False,
            "error_msg": None
        }

        if not answer_str:
            result["error_msg"] = "Tag <answer> not found or empty"
            return result

        # 1. Parse AST and enforce security whitelist
        try:
            clean_expr = answer_str.split("=")[0].strip()
            tree = ast.parse(clean_expr, mode='eval')
            if not cls.is_safe_ast(tree):
                result["error_msg"] = "Expression contains unallowed elements (only +, -, *, /, parentheses, and numbers)"
                return result
            result["valid_syntax"] = True
        except Exception as e:
            result["error_msg"] = f"AST syntax error: {e}"
            return result

        # 2. Check number usage constraints
        used_numbers = cls.extract_numbers_from_ast(tree)
        result["used_numbers"] = used_numbers

        available_copy = list(available_numbers)
        valid_nums = True
        for num in used_numbers:
            if num in available_copy:
                available_copy.remove(num)
            else:
                valid_nums = False
                break

        result["valid_numbers"] = valid_nums and (len(used_numbers) > 0)
        if not result["valid_numbers"]:
            result["error_msg"] = f"Invalid numbers or numbers used more than once: {used_numbers} (Available: {available_numbers})"
            return result

        # 3. Evaluate expression
        computed = cls.evaluate_ast(tree)
        result["computed_value"] = computed

        if computed is None:
            result["error_msg"] = "Evaluation failed (e.g. division by zero)"
            return result

        # 4. Check target equality (with float tolerance for exact divisions)
        if abs(computed - target) < 1e-6:
            result["is_correct"] = True
        else:
            result["error_msg"] = f"Result {computed} != target {target}"

        return result

## 3. Load Base Model (Qwen2.5-0.5B-Instruct)

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

## 4. Load Benchmark Test Set (Auto-Download from GitHub)

In [ ]:
data_dir = "data" if not os.path.basename(os.getcwd()) == "notebooks" else "../data"
os.makedirs(data_dir, exist_ok=True)
test_file = os.path.join(data_dir, "test_countdown.jsonl")

# Automatically fetch the 200 benchmark test puzzles from the repository if not found locally
if not os.path.exists(test_file):
    print(f"'{test_file}' not found locally. Downloading the 200 test puzzles from GitHub...")
    url = "https://raw.githubusercontent.com/Gheb6/countdown-grpo-reasoning/main/data/test_countdown.jsonl"
    urllib.request.urlretrieve(url, test_file)

with open(test_file, "r", encoding="utf-8") as f:
    test_samples = [json.loads(line) for line in f]

print(f"✅ Loaded {len(test_samples)} benchmark test puzzles from {test_file}!")
print("Sample puzzle #1:", test_samples[0])

## 5. Benchmark Execution: Zero-Shot vs CoT

In [ ]:
def evaluate_prompt_mode(mode="zero_shot", samples=test_samples):
    correct_count = 0
    total_tokens = 0
    results = []

    for item in tqdm(samples, desc=f"Evaluating {mode}"):
        nums = item["numbers"]
        target = item["target"]

        if mode == "zero_shot":
            user_msg = f"Using numbers {nums}, create an equation that equals {target}. Do not explain. Output ONLY the equation in <answer>...</answer>."
        else: # Chain of Thought
            user_msg = f"Using numbers {nums}, create an equation that equals {target}. You can use (+, -, *, /) at most once per number. Think step by step inside <think>...</think> and output the final equation in <answer>...</answer>."

        messages = [
            {"role": "system", "content": "You are a helpful mathematical assistant."},
            {"role": "user", "content": user_msg}
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=384 if mode == "cot" else 64,
                do_sample=False, # deterministic greedy decoding
                pad_token_id=tokenizer.eos_token_id
            )

        gen_tokens = output_ids[0][inputs.input_ids.shape[1]:]
        completion = tokenizer.decode(gen_tokens, skip_special_tokens=True)
        total_tokens += len(gen_tokens)

        verif = CountdownVerifier.verify(nums, target, completion)
        if verif["is_correct"]:
            correct_count += 1

        results.append({
            "target": target,
            "numbers": nums,
            "completion": completion,
            "is_correct": verif["is_correct"],
            "tokens": len(gen_tokens)
        })

    acc = (correct_count / len(samples)) * 100
    avg_len = total_tokens / len(samples)
    print(f"\n--- Results for {mode.upper()} ---")
    print(f"Accuracy: {acc:.2f}% ({correct_count}/{len(samples)})")
    print(f"Average Completion Length: {avg_len:.1f} tokens")
    return acc, avg_len, results

# Evaluate on a representative subset of the benchmark (e.g. first 50 or full 200)
BENCHMARK_SIZE = 50
eval_subset = test_samples[:BENCHMARK_SIZE]

print(f"1. Evaluating Method 1: Zero-Shot Direct on {len(eval_subset)} samples...")
zs_acc, zs_len, zs_res = evaluate_prompt_mode("zero_shot", samples=eval_subset)

print(f"\n2. Evaluating Method 2: Chain-of-Thought (CoT) on {len(eval_subset)} samples...")
cot_acc, cot_len, cot_res = evaluate_prompt_mode("cot", samples=eval_subset)

## 6. Baseline Results & Metrics Export

In [ ]:
df = pd.DataFrame([
    {"Methodology": "1. Zero-Shot Direct", "Accuracy (%)": f"{zs_acc:.2f}%", "Avg Tokens": f"{zs_len:.1f}"},
    {"Methodology": "2. Chain-of-Thought (CoT)", "Accuracy (%)": f"{cot_acc:.2f}%", "Avg Tokens": f"{cot_len:.1f}"}
])
print(df.to_string(index=False))

res_dir = "results" if not os.path.basename(os.getcwd()) == "notebooks" else "../results"
os.makedirs(res_dir, exist_ok=True)
output_path = os.path.join(res_dir, "baselines.json")

with open(output_path, "w") as f:
    json.dump({
        "zero_shot_accuracy": zs_acc,
        "zero_shot_tokens": zs_len,
        "cot_accuracy": cot_acc,
        "cot_tokens": cot_len,
        "benchmark_samples": len(eval_subset)
    }, f, indent=2)

print(f"Baselines successfully saved to {output_path}")